# Appendix A1. The Same Model in Four Tools: Excel Solver, Pyomo, JuMP and GAMS

**2105623 Optimization of Chemical Processes** | Assoc. Prof. Dr. Soorathep Kheawhom
Department of Chemical Engineering, Chulalongkorn University | Semester 1/2026

**Appendix.** Read in Week 1, or as self-study at any later point.

**CLO mapping:** CLO 4 (implement and solve optimization models with computational tools, handling data
input, model modification and solution interpretation).

## Learning objectives

- Identify, in each of four tools, the six constructs that every algebraic model needs: index set,
  parameter, variable with bounds, linear constraint over a set, objective, and solve and retrieve.
- Lay out a small linear program on a spreadsheet, say exactly which cells hold the decision variables,
  the objective and the constraint left-hand sides, and fill in the Excel Solver dialog correctly.
- Build the same model in Pyomo, check the termination condition, and read the primal values and the
  shadow prices.
- Read a JuMP or a GAMS model written by somebody else and translate it back into algebra, or forward
  into Pyomo, without needing to run it.
- State what the choice of modeling language does not change (the optimal solution) and what it does
  change (ecosystem, model generation speed, licensing, auditability, stakeholder access).

**Estimated duration:** 20 to 30 minutes.

**Prerequisites:** Week 1 (the optimization workflow, the modeling layer versus the solver layer, and a
first Pyomo model).

**Execution policy.** Only the Pyomo section executes. Julia and GAMS are not course requirements and
neither runtime is installed here, so the JuMP and GAMS models appear as formatted source listings with
commentary. The Pyomo section therefore produces the reference answer that the other three sections are
checked against.

**Why this appendix exists.** Pyomo is the single language of instruction in this course. That is a
teaching decision, not a claim about the mathematics. A linear program is a matrix, a right-hand side and
a cost vector; the tool only decides how those objects are typed in. A graduate who meets GAMS in a
refinery planning group or JuMP in a research code should recognize the same model wearing different
clothes, and this appendix is the demonstration.

**Reference:** Williams, *Model Building in Mathematical Programming*, Ch. 1 and Ch. 3; Rao, Ch. 3.

In [ ]:
# --- Environment check -------------------------------------------------------
import sys, subprocess, importlib, shutil

def ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg])

for p, n in [("pyomo", "pyomo"), ("numpy", "numpy"), ("scipy", "scipy"),
             ("matplotlib", "matplotlib"), ("pandas", "pandas")]:
    ensure(p, n)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pyomo.environ as pyo

def pick_solver(kind="lp"):
    """Return the first available solver of the requested kind."""
    order = {"lp":   ["appsi_highs", "glpk", "cbc", "gurobi", "cplex"],
             "milp": ["appsi_highs", "cbc", "glpk", "gurobi", "cplex"],
             "nlp":  ["ipopt", "conopt", "knitro"],
             "minlp":["bonmin", "couenne", "mindtpy"]}[kind]
    for name in order:
        try:
            s = pyo.SolverFactory(name)
            if s is not None and s.available(exception_flag=False):
                print(f"Using solver: {name}")
                return s
        except Exception:
            continue
    raise RuntimeError(f"No {kind} solver found. Install one, e.g. 'pip install highspy' "
                       f"or 'conda install -c conda-forge ipopt glpk coincbc'.")

In [ ]:
# --- Figure style: Teal-Amber Lab Palette v1.0 -------------------------------
PALETTE = ["#0F6E6B", "#E29A2D", "#BE654C", "#5A91BE", "#83A462", "#995A90", "#333F4A", "#DFC98F"]
INK, GRAPHITE, MIST, PAPER = "#1C242B", "#333F4A", "#B9C1C6", "#F3F0EB"

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRAPHITE, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.linewidth": 1.0, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": MIST, "grid.linewidth": 0.7, "grid.alpha": 0.9,
    "xtick.color": GRAPHITE, "ytick.color": GRAPHITE,
    "text.color": INK, "lines.linewidth": 1.8, "lines.markersize": 5,
    "font.size": 9, "legend.frameon": False,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
})

def tidy(ax):
    """Apply the house style to a single Axes object."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRAPHITE)
    return ax

print("Palette loaded:", ", ".join(PALETTE[:3]), "...")

## 1. One model, stated once

A specialty chemical unit supplying a battery plant makes three products on shared equipment during a
one-week planning horizon:

- **E1**, a standard carbonate electrolyte,
- **E2**, a high-purity electrolyte for cells with tighter moisture specifications,
- **E3**, an additive concentrate sold to other formulators.

Three resources are shared: a **reactor** (mixing and reaction), a **purification** train (distillation
and drying), and a **packaging** line (filling and quality control). Everything produced in the week can
be sold, and finished-goods tankage limits any single product to 25 t per week.

### Algebraic model

$$\begin{aligned}
\max_{x} \quad & z = \sum_{p \in P} c_p \, x_p && \text{(contribution margin, kUSD/week)}\\
\text{s.t.}\quad & \sum_{p \in P} a_{rp} \, x_p \le b_r && \forall r \in R \quad \text{(resource hours)}\\
& 0 \le x_p \le u_p && \forall p \in P \quad \text{(tankage limit, non-negativity)}
\end{aligned}$$

### Symbol table

| Symbol | Meaning | Units | Role |
|---|---|---|---|
| `p in P` | product, `P = {E1, E2, E3}` | - | index set |
| `r in R` | shared resource, `R = {reactor, purification, packaging}` | - | index set |
| `c_p` | contribution margin of product `p` | kUSD/t | parameter |
| `a_rp` | hours of resource `r` consumed per tonne of product `p` | h/t | parameter |
| `b_r` | hours of resource `r` available in the week | h/week | parameter |
| `u_p` | maximum tonnes of product `p` the tankage can hold | t/week | parameter (bound) |
| `x_p` | tonnes of product `p` made in the week | t/week | decision variable |
| `z` | total weekly contribution margin | kUSD/week | objective |

This is a linear program: the objective and every constraint are linear in `x`, and `x` is continuous.
Three variables and three structural constraints, so it can be solved by hand.

### The reference answer

The instance below has a unique, non-degenerate optimum:

- `x = (E1, E2, E3) = (20, 15, 10)` t/week,
- `z = 121.5` kUSD/week,
- shadow prices `y = (reactor, purification, packaging) = (0.50, 0.40, 0.30)` kUSD/h,
- all three resource constraints tight, and every tankage bound `u_p = 25` slack.

Every one of the four tools below must return exactly these numbers. Section 3 computes them; Section 3.4
also reproduces them without any solver at all, by solving the 3 by 3 system of tight constraints, so that
the reference answer does not depend on any single piece of software.

In [ ]:
# --- The one instance used by all four tools ---------------------------------
PRODUCTS  = ["E1", "E2", "E3"]
RESOURCES = ["reactor", "purification", "packaging"]

margin = {"E1": 2.5, "E2": 3.1, "E3": 2.5}                     # c_p  [kUSD/t]
avail  = {"reactor": 130.0, "purification": 100.0, "packaging": 55.0}   # b_r [h/week]
tank   = {"E1": 25.0, "E2": 25.0, "E3": 25.0}                  # u_p  [t/week]

use = {("reactor",      "E1"): 2.0, ("reactor",      "E2"): 4.0, ("reactor",      "E3"): 3.0,
       ("purification", "E1"): 3.0, ("purification", "E2"): 2.0, ("purification", "E3"): 1.0,
       ("packaging",    "E1"): 1.0, ("packaging",    "E2"): 1.0, ("packaging",    "E3"): 2.0}  # a_rp [h/t]

tech = pd.DataFrame([[use[r, p] for p in PRODUCTS] for r in RESOURCES],
                    index=RESOURCES, columns=PRODUCTS)
tech["hours available [h/week]"] = [avail[r] for r in RESOURCES]

print("resource use a_rp [h/t] and availability b_r [h/week]")
print(tech.to_string())
print("\ncontribution margin c_p [kUSD/t] :", margin)
print("tankage limit        u_p [t/week] :", tank)

## 2. Excel Solver

Excel is a modeling layer like any other. The difference is that its "code" lives in cell formulas, so the
model and the data occupy the same object, and the model is read by looking at a grid rather than at a
listing.

The convention used here is the standard one: **one column per product, one row per resource**, decision
variables in a single contiguous row so that they can be selected as one range, and every constraint left
hand side computed with `SUMPRODUCT` against an absolute reference to that row.

The layout below is what would be typed into a blank worksheet. Formulas are shown as text.

In [ ]:
# --- The worksheet, cell by cell ---------------------------------------------
grid = {c: [""] * 12 for c in list("ABCDEFG")}

def put(cell, value):
    col, row = cell[0], int(cell[1:])
    grid[col][row - 1] = value

put("A1", "Weekly product mix, electrolyte unit")
put("A2", "product");                 put("B2", "E1");   put("C2", "E2");   put("D2", "E3")
put("A3", "margin c_p [kUSD/t]");     put("B3", "2.5");  put("C3", "3.1");  put("D3", "2.5")
put("A4", "tonnes x_p [t/week]");     put("B4", "0");    put("C4", "0");    put("D4", "0")
put("A5", "tank limit u_p [t/week]"); put("B5", "25");   put("C5", "25");   put("D5", "25")

put("A7", "resource");     put("B7", "E1"); put("C7", "E2"); put("D7", "E3")
put("E7", "hours used");   put("F7", "");   put("G7", "hours available")
for k, (res, row) in enumerate(zip(RESOURCES, [8, 9, 10])):
    put(f"A{row}", res)
    for col, p in zip("BCD", PRODUCTS):
        put(f"{col}{row}", f"{use[res, p]:g}")
    put(f"E{row}", f"=SUMPRODUCT(B{row}:D{row},$B$4:$D$4)")
    put(f"F{row}", "<=")
    put(f"G{row}", f"{avail[res]:g}")

put("A12", "total margin z [kUSD/week]")
put("B12", "=SUMPRODUCT(B3:D3,B4:D4)")

sheet = pd.DataFrame(grid, index=[str(i) for i in range(1, 13)])
sheet.index.name = "row"
with pd.option_context("display.max_colwidth", 40, "display.width", 200):
    print(sheet.to_string())

In [ ]:
# --- Which cells play which role ---------------------------------------------
roles = pd.DataFrame(
    [["$B$4:$D$4", "decision variables x_p", "changing cells: numbers, never formulas",
      "3 cells"],
     ["$B$12",     "objective z",            "=SUMPRODUCT(B3:D3,B4:D4)", "1 cell"],
     ["$E$8:$E$10","constraint left-hand sides sum_p a_rp x_p",
      "=SUMPRODUCT(B8:D8,$B$4:$D$4) filled down", "3 cells"],
     ["$G$8:$G$10","constraint right-hand sides b_r", "data: hours available", "3 cells"],
     ["$B$3:$D$3", "parameters c_p",   "data: contribution margin",   "3 cells"],
     ["$B$8:$D$10","parameters a_rp",  "data: the technology matrix", "9 cells"],
     ["$B$5:$D$5", "parameters u_p",   "data: upper bounds, entered as a constraint row", "3 cells"]],
    columns=["range", "role in the algebraic model", "what it contains", "size"])
with pd.option_context("display.max_colwidth", 48, "display.width", 200):
    print(roles.to_string(index=False))

print("\nRule: a cell is either data, or a formula, never both. "
      "A number typed into a formula is the spreadsheet form of a hard-coded constant.")

### The Solver dialog, field by field

`Data > Solver` opens the dialog. Filled in for this worksheet, it reads:

| Field | Entry |
|---|---|
| Set Objective | `$B$12` |
| To | `Max` |
| By Changing Variable Cells | `$B$4:$D$4` |
| Subject to the Constraints | `$E$8:$E$10 <= $G$8:$G$10` (resource hours) |
| | `$B$4:$D$4 <= $B$5:$D$5` (tankage limit) |
| Make Unconstrained Variables Non-Negative | checked (this supplies `x_p >= 0`) |
| Select a Solving Method | `Simplex LP` |
| Options | defaults; on the results dialog select the **Sensitivity** report to read shadow prices |

Three points about this dialog that cost students time.

- **`Simplex LP` is the correct engine, not `GRG Nonlinear`.** GRG will usually find the same point on a
  linear model, but it is a local nonlinear method: it gives no proof of optimality, no simplex-based
  sensitivity report, and it can stop early. Choose the engine that matches the model class.
- **The non-negativity checkbox is a constraint.** If it is cleared, Excel allows negative production and
  the answer changes. It is easy to clear it by accident and never notice.
- **Constraints are entered as ranges, not one row at a time.** `$E$8:$E$10 <= $G$8:$G$10` is one dialog
  entry standing for three constraints. This is the spreadsheet equivalent of indexing a constraint over a
  set, and it is the only place where Excel expresses the idea of `for all r in R`.

**Answer.** Solver returns `$B$4:$D$4 = 20, 15, 10` and `$B$12 = 121.5`. The Sensitivity report gives
shadow prices `0.50, 0.40, 0.30` kUSD/h on the three resource rows, and slack on both tankage bounds and
zero reduced cost on all three products.

**Where Excel stops.** Standard Excel Solver is limited to 200 changing cells and 100 explicit constraints,
the model cannot be diffed or unit tested because it is stored in cell formulas, and the structure is
implicit: nothing on the worksheet states that column C is the same product as column C in the block
below. OpenSolver removes the size limit by sending the same worksheet to CBC or HiGHS, but not the other
two problems.

## 3. Pyomo

The same model as a Pyomo `ConcreteModel`. Each algebraic object of Section 1 becomes exactly one named
component, which is the whole point of an algebraic modeling language: the code is readable as algebra,
and the data enters through `initialize` rather than being written into the constraint expressions.

This is the only section of the notebook that runs.

In [ ]:
# --- 3.1 The model -----------------------------------------------------------
m = pyo.ConcreteModel(name="product mix, electrolyte unit")

m.P = pyo.Set(initialize=PRODUCTS)                                  # index set: products
m.R = pyo.Set(initialize=RESOURCES)                                 # index set: resources

m.c = pyo.Param(m.P, initialize=margin)                             # margin      [kUSD/t]
m.b = pyo.Param(m.R, initialize=avail)                              # available   [h/week]
m.u = pyo.Param(m.P, initialize=tank)                               # tank limit  [t/week]
m.a = pyo.Param(m.R, m.P, initialize=use)                           # technology  [h/t]

m.x = pyo.Var(m.P, domain=pyo.NonNegativeReals,                     # decision variable with bounds
              bounds=lambda mm, p: (0.0, mm.u[p]))

def capacity_rule(mm, r):                                           # one constraint per resource
    return sum(mm.a[r, p] * mm.x[p] for p in mm.P) <= mm.b[r]
m.capacity = pyo.Constraint(m.R, rule=capacity_rule)

m.z = pyo.Objective(expr=sum(m.c[p] * m.x[p] for p in m.P),         # objective
                    sense=pyo.maximize)

m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)                    # ask the solver for duals

print(f"variables   : {sum(1 for _ in m.component_data_objects(pyo.Var))}")
print(f"constraints : {sum(1 for _ in m.component_data_objects(pyo.Constraint))}")
print(f"sense       : {'maximize' if m.z.sense == pyo.maximize else 'minimize'}")

In [ ]:
# --- 3.2 Solve and check -----------------------------------------------------
lp = pick_solver("lp")
res = lp.solve(m)
assert res.solver.termination_condition == pyo.TerminationCondition.optimal, \
    f"solver did not prove optimality: {res.solver.termination_condition}"

z_star = pyo.value(m.z)
x_star = {p: pyo.value(m.x[p]) for p in PRODUCTS}
y_star = {r: m.dual[m.capacity[r]] for r in RESOURCES}

print(f"termination     : {res.solver.termination_condition}")
print(f"optimal margin z: {z_star:.4f} kUSD/week")
print("optimal plan x_p: " + ", ".join(f"{p} = {x_star[p]:.4f} t/week" for p in PRODUCTS))

In [ ]:
# --- 3.3 Results tables ------------------------------------------------------
plan = pd.DataFrame({
    "tonnes x_p [t/week]":  [x_star[p] for p in PRODUCTS],
    "margin c_p [kUSD/t]":  [margin[p] for p in PRODUCTS],
    "contribution [kUSD/week]": [margin[p] * x_star[p] for p in PRODUCTS],
    "tank limit u_p [t/week]":  [tank[p] for p in PRODUCTS],
    "bound status": ["slack" if x_star[p] < tank[p] - 1e-9 else "at bound" for p in PRODUCTS],
}, index=PRODUCTS)
print(plan.to_string(float_format=lambda v: f"{v:.4g}"))
print(f"total contribution = {z_star:.4g} kUSD/week")

used = {r: sum(use[r, p] * x_star[p] for p in PRODUCTS) for r in RESOURCES}
resource_tbl = pd.DataFrame({
    "hours used": [used[r] for r in RESOURCES],
    "hours available b_r": [avail[r] for r in RESOURCES],
    "slack": [avail[r] - used[r] for r in RESOURCES],
    "shadow price [kUSD/h]": [y_star[r] for r in RESOURCES],
}, index=RESOURCES)
print("\n" + resource_tbl.to_string(float_format=lambda v: f"{v:.4g}"))
print("\nAll three resources are tight, so all three shadow prices are strictly positive.")

In [ ]:
# --- 3.4 The same answer without a solver ------------------------------------
# All three resource constraints are tight at the optimum and all three products are made,
# so x solves the 3 by 3 linear system A x = b, and the duals solve A' y = c.
A = np.array([[use[r, p] for p in PRODUCTS] for r in RESOURCES], dtype=float)
bvec = np.array([avail[r] for r in RESOURCES], dtype=float)
cvec = np.array([margin[p] for p in PRODUCTS], dtype=float)

x_hand = np.linalg.solve(A, bvec)          # primal, from the tight constraints
y_hand = np.linalg.solve(A.T, cvec)        # dual,   from the reduced-cost conditions

print("A x = b  gives x =", np.round(x_hand, 10), "t/week")
print("A' y = c gives y =", np.round(y_hand, 10), "kUSD/h")
print(f"primal objective c'x = {cvec @ x_hand:.6f} kUSD/week")
print(f"dual   objective b'y = {bvec @ y_hand:.6f} kUSD/week   (strong duality: the two agree)")

assert np.allclose(x_hand, [x_star[p] for p in PRODUCTS], atol=1e-8)
assert np.allclose(y_hand, [y_star[r] for r in RESOURCES], atol=1e-8)
assert np.all(x_hand > 0) and np.all(y_hand > 0), "expected a non-degenerate interior basis"
print("\nSolver and hand calculation agree, and x > 0 with y > 0 means the optimal basis is unique:")
print("no alternative optimum exists, so any correct LP solver must return exactly this vertex.")

In [ ]:
# --- Figure: where the hours go ----------------------------------------------
fig, ax = plt.subplots(figsize=(6.2, 3.0))
ypos = np.arange(len(RESOURCES))[::-1]
left = np.zeros(len(RESOURCES))
for k, p in enumerate(PRODUCTS):
    width = np.array([use[r, p] * x_star[p] for r in RESOURCES])
    ax.barh(ypos, width, left=left, height=0.55, color=PALETTE[k], edgecolor="white",
            linewidth=0.8, label=f"{p} ({x_star[p]:.0f} t/week)")
    left += width
ax.plot([avail[r] for r in RESOURCES], ypos, linestyle="none", marker="D",
        color=PALETTE[2], markersize=6, label="hours available b_r")
ax.set_yticks(ypos); ax.set_yticklabels(RESOURCES)
ax.set_xlabel("resource hours consumed in the week [h]")
ax.set_title(f"Optimal plan, z = {z_star:.1f} kUSD/week (identical in all four tools)")
ax.legend(fontsize=7.5, loc="lower right")
tidy(ax); fig.tight_layout(); plt.show()

**Reading the result.** Every resource is fully consumed, which is why every shadow price is positive: one
extra reactor hour is worth 0.50 kUSD, one extra purification hour 0.40 kUSD, one extra packaging hour
0.30 kUSD. The tankage bounds `u_p = 25` never bind, so they cost nothing here; they are kept in the model
because they are physically real, and because they give each of the four tools something to say about
bounded variables.

Everything from here on is reading, not running. The listings in Sections 4 and 5 are checked against
`z = 121.5` and `x = (20, 15, 10)`.

## 4. JuMP (Julia)

JuMP is the algebraic modeling layer of the Julia ecosystem. Structurally it is very close to Pyomo: a
model object, components attached to it, a solver called through a common interface (MathOptInterface,
the Julia analogue of Pyomo's solver plugins). The listing below is deliberately written line for line
against Section 3.1, in the same order, with the same names.

```julia
using JuMP, HiGHS                                        # modeling layer + solver

P = ["E1", "E2", "E3"]                                   # index set: products
R = ["reactor", "purification", "packaging"]             # index set: resources

c = Dict("E1" => 2.5, "E2" => 3.1, "E3" => 2.5)          # margin     [kUSD/t]
b = Dict("reactor" => 130.0,                             # available  [h/week]
         "purification" => 100.0,
         "packaging" => 55.0)
u = Dict("E1" => 25.0, "E2" => 25.0, "E3" => 25.0)       # tank limit [t/week]
a = Dict(("reactor", "E1")      => 2.0, ("reactor", "E2")      => 4.0, ("reactor", "E3")      => 3.0,
         ("purification", "E1") => 3.0, ("purification", "E2") => 2.0, ("purification", "E3") => 1.0,
         ("packaging", "E1")    => 1.0, ("packaging", "E2")    => 1.0, ("packaging", "E3")    => 2.0)

model = Model(HiGHS.Optimizer)                           # ConcreteModel + solver in one object

@variable(model, 0 <= x[p in P] <= u[p])                 # decision variable with bounds

@constraint(model, capacity[r in R],                     # one constraint per resource
            sum(a[r, p] * x[p] for p in P) <= b[r])

@objective(model, Max, sum(c[p] * x[p] for p in P))      # objective

optimize!(model)                                         # solve, in place

@assert termination_status(model) == OPTIMAL             # status check, always
println("optimal margin z = ", objective_value(model), " kUSD/week")
for p in P
    println("  x[", p, "] = ", value(x[p]), " t/week")
end
for r in R
    println("  shadow price ", r, " = ", shadow_price(capacity[r]), " kUSD/h")
end
```

**Output.** `z = 121.5`, `x = (20.0, 15.0, 10.0)`, shadow prices `(0.5, 0.4, 0.3)`: identical to Section 3.

### What is different, syntactically

- **Macros.** `@variable`, `@constraint` and `@objective` are macros, not functions. They rewrite the
  Julia expression at parse time, which is why `0 <= x[p in P] <= u[p]` can be written as mathematics
  rather than as a `bounds=` keyword taking a function. The leading `@` is the visible marker that the
  expression is being transformed rather than evaluated.
- **Container syntax.** `x[p in P]` declares an indexed family in one line and returns a container keyed
  by the elements of `P` (a `DenseAxisArray`), so `x["E2"]` is a variable and `x` is the family. Pyomo
  reaches the same place through `pyo.Var(m.P)`; the difference is that in JuMP the index set is an
  ordinary Julia collection, while in Pyomo it is usually declared as a `Set` component of the model.
- **Named constraint families.** `@constraint(model, capacity[r in R], ...)` gives the family the name
  `capacity`, indexed the same way as `m.capacity` in Pyomo. That name is what `shadow_price` is later
  applied to.
- **`optimize!` with an exclamation mark.** Julia convention marks functions that mutate their argument
  with a trailing `!`. `optimize!(model)` stores the solution inside `model`; it does not return a results
  object the way `solver.solve(m)` does in Pyomo. Status and values are then queried with
  `termination_status`, `objective_value`, `value` and `shadow_price`.
- **Broadcasting with a dot.** `value(x[p])` retrieves one number; `value.(x)` (note the dot) broadcasts
  `value` over the whole container and returns the family of numbers at once. The dot is Julia's general
  elementwise operator and appears everywhere, not just in JuMP.
- **Duals and sign conventions.** `shadow_price` is used above rather than `dual` because it is defined as
  the improvement in the objective per unit increase in the right-hand side, which has the same sign for
  maximization and minimization. `dual` follows the MathOptInterface convention, whose sign depends on the
  sense of the problem, and is a standard source of confusion when comparing tools.

### Why you should believe it gives the same answer

Nothing in the list above touches the model. Both programs declare the same two index sets, the same
numerical data, the same bounds `0 <= x_p <= 25`, the same three inequalities `sum_p a_rp x_p <= b_r`, and
the same linear objective with `Max`. Both then hand the resulting matrix, right-hand side and cost vector
to a simplex or interior point implementation (HiGHS in both cases, if `appsi_highs` was selected in
Section 3.2). Section 3.4 showed that the optimal basis is unique and non-degenerate, so the LP has one
optimal vertex and no alternative optima: any correct solver, reached through any correct modeling layer,
must return `x = (20, 15, 10)` and `z = 121.5`. Believing this does not require running the Julia code, it
requires checking that the listing declares the same mathematical objects, which is what the line-for-line
layout is for. That check is the reader's job whenever a model is ported between tools.

## 5. GAMS

GAMS is the oldest of the four and remains common in refinery, petrochemical and energy planning groups.
It is a standalone declarative language rather than a library inside a general-purpose language, which
makes it more different from Pyomo than JuMP is, and yet the six constructs are all still there.

```gams
$title Weekly product mix, electrolyte unit

Sets
   p   products    / E1, E2, E3 /
   r   resources   / reactor, purification, packaging / ;

Parameters
   c(p)   contribution margin in kUSD per tonne
            / E1 2.5, E2 3.1, E3 2.5 /
   b(r)   hours available per week
            / reactor 130, purification 100, packaging 55 /
   u(p)   tankage limit in tonnes per week
            / E1 25, E2 25, E3 25 / ;

Table a(r,p)   resource hours per tonne
                        E1     E2     E3
   reactor               2      4      3
   purification          3      2      1
   packaging             1      1      2   ;

Positive Variable  x(p)   tonnes of product p per week ;
Variable           z      total contribution margin in kUSD per week ;

x.up(p) = u(p) ;

Equations
   margindef        definition of the objective
   capacity(r)      resource hours available ;

margindef ..       z =e= sum(p, c(p)*x(p)) ;
capacity(r) ..     sum(p, a(r,p)*x(p)) =l= b(r) ;

Model productmix / all / ;
solve productmix using lp maximizing z ;

display x.l, z.l, capacity.m ;
```

**Output.** `x.l = (20, 15, 10)`, `z.l = 121.5`, `capacity.m = (0.5, 0.4, 0.3)`: identical to Section 3.

### What is different, syntactically

- **Sets are declared, with their elements inline.** `Set p / E1, E2, E3 /` declares both the set and its
  members between slashes. The text between the identifier and the slashes is documentation that GAMS
  carries into the listing file. There is no equivalent of building a set from a computed Python list;
  data normally arrives from an include file, a spreadsheet or a GDX database.
- **Parameters are indexed data, and `Table` is the two-dimensional form.** `c(p)` and `b(r)` take
  slash-delimited value lists. A two-index parameter such as `a(r,p)` is more readable as a `Table`, which
  is laid out exactly like the technology matrix printed in Section 1. Unlisted entries default to zero,
  which is convenient for sparse data and dangerous when a typo silently becomes a zero.
- **Variable type carries the bound.** `Positive Variable x(p)` supplies `x_p >= 0` at declaration.
  Other bounds are assigned afterwards through suffixes: `x.up(p) = u(p)` sets the upper bound, `x.lo`
  the lower bound, `x.fx` fixes a variable. This is an assignment statement, not part of the declaration,
  so bounds can be recomputed between solves.
- **The objective is a free variable defined by an equation.** GAMS has no `Objective` component. You
  declare a free scalar variable `z`, define it with an equality equation (`margindef .. z =e= ...`), and
  then name `z` in the `solve` statement. Forgetting to declare `z` as free, or declaring it positive, is
  a classic error that makes a model infeasible for no visible reason.
- **Declaration and definition of equations are separate.** The `Equations` block declares the names and
  their index sets; the definitions follow, each written as `name(index) .. body ;` with `..` separating
  the name from the body. The relational operators are `=e=` for equality, `=l=` for less than or equal
  and `=g=` for greater than or equal. Ordinary `=` is reserved for assignment to data, so the model
  algebra and the data algebra can never be confused.
- **`solve ... using lp maximizing z`.** One statement names the model, the problem class and the
  objective variable, and the sense. The class is declared by the modeler, not inferred: writing `lp` on a
  model that contains a product of variables is an error GAMS will report, and writing `nlp` on a linear
  model is legal but sends it to a nonlinear solver. `Model productmix / all /` collects every declared
  equation; a subset can be listed instead to solve a variant of the same model.
- **Results are read through suffixes.** `.l` is the level (the primal value), `.m` the marginal (the dual
  for an equation, the reduced cost for a variable), `.lo` and `.up` the bounds. `display` writes them to
  the listing file. There is one result object per symbol rather than a separate results structure.

### Why you should believe it gives the same answer

The same argument as for JuMP. The `Table` is the matrix `a_rp` of Section 1 transcribed row by row, the
parameter lists are `c_p`, `b_r` and `u_p`, `Positive Variable` plus `x.up` is `0 <= x_p <= 25`,
`capacity(r) .. =l=` is the resource constraint family, and `margindef` with `maximizing z` is the
objective. The LP handed to the solver is therefore the same matrix, and by Section 3.4 that LP has a
unique optimal vertex.

## 6. Side by side

Six constructs are enough to write any linear or mixed-integer model. This table is the translation
dictionary for the whole appendix.

| Construct | Excel Solver | Pyomo | JuMP | GAMS |
|---|---|---|---|---|
| **Index set** | implicit: a contiguous block, products in columns B to D, resources in rows 8 to 10; the set exists only as a geometric convention | `m.P = pyo.Set(initialize=["E1","E2","E3"])` | `P = ["E1", "E2", "E3"]`, an ordinary Julia vector used inside the macros | `Set p products / E1, E2, E3 / ;` |
| **Parameter** | a data cell or range, `B3:D3` for `c_p`, `B8:D10` for `a_rp` | `m.c = pyo.Param(m.P, initialize=margin)` | `c = Dict("E1" => 2.5, "E2" => 3.1, "E3" => 2.5)` | `Parameter c(p) / E1 2.5, E2 3.1, E3 2.5 / ;` or `Table a(r,p)` |
| **Variable with bounds** | changing cells `$B$4:$D$4`, upper bounds as the dialog row `$B$4:$D$4 <= $B$5:$D$5`, lower bound from the non-negativity checkbox | `m.x = pyo.Var(m.P, domain=pyo.NonNegativeReals, bounds=lambda mm, p: (0, mm.u[p]))` | `@variable(model, 0 <= x[p in P] <= u[p])` | `Positive Variable x(p) ;` then `x.up(p) = u(p) ;` |
| **Linear constraint over a set** | `=SUMPRODUCT(B8:D8,$B$4:$D$4)` in `E8`, filled down to `E10`, plus the dialog entry `$E$8:$E$10 <= $G$8:$G$10` | `m.capacity = pyo.Constraint(m.R, rule=capacity_rule)` with `return sum(mm.a[r,p]*mm.x[p] for p in mm.P) <= mm.b[r]` | `@constraint(model, capacity[r in R], sum(a[r,p]*x[p] for p in P) <= b[r])` | `capacity(r) .. sum(p, a(r,p)*x(p)) =l= b(r) ;` |
| **Objective** | `=SUMPRODUCT(B3:D3,B4:D4)` in `B12`, entered as Set Objective with To: Max | `m.z = pyo.Objective(expr=sum(m.c[p]*m.x[p] for p in m.P), sense=pyo.maximize)` | `@objective(model, Max, sum(c[p]*x[p] for p in P))` | free variable `z` plus `margindef .. z =e= sum(p, c(p)*x(p)) ;` |
| **Solve and retrieve** | press Solve with the `Simplex LP` engine; values overwrite `$B$4:$D$4`; duals come from the Sensitivity report | `res = lp.solve(m)`, check `res.solver.termination_condition`, then `pyo.value(m.x[p])` and `m.dual[m.capacity[r]]` | `optimize!(model)`, check `termination_status(model)`, then `value.(x)` and `shadow_price(capacity[r])` | `solve productmix using lp maximizing z ;` then `x.l(p)`, `z.l`, `capacity.m(r)`, and check `productmix.modelstat` |

Two observations worth carrying away.

- Every tool except Excel has an explicit object for the index set, and that is the single largest
  practical difference. In the other three, adding a fourth product is one line of data. In Excel it is a
  new column plus an edit to every formula and every dialog range that referred to the old block.
- Every tool has a status to check, and in all four it is possible to read numbers off a model that was
  never solved to optimality. `res.solver.termination_condition`, `termination_status(model)`,
  `productmix.modelstat` and the sentence at the top of the Excel Solver Results dialog are the same
  safeguard in four spellings. Reporting a solution without checking it is the same error in every tool.

## 7. When to reach for which

All four produce the same answer on this model, so the choice is made on everything except the arithmetic.

- **Excel Solver, for small stakeholder-facing models.** Its advantage is that the audience already has it
  and can see the data. A production planner who will not open a terminal will open a workbook, change a
  number and re-solve. Use it when the model is small (well inside 200 changing cells and 100 explicit
  constraints), when the model is unlikely to change structurally, and when the people who must trust the
  answer need to see the arithmetic. Its limits are size, the absence of an explicit index set, the
  impossibility of version control or unit testing on formulas, and an LP and NLP engine pair that is far
  weaker than modern solvers. OpenSolver removes the size limit and substitutes CBC or HiGHS, and keeps
  everything else.

- **Pyomo, for research work and anything that lives in the Python ecosystem.** The model is ordinary
  Python, so data preparation with pandas, plotting, statistics, machine learning, unit tests and version
  control are all available without leaving the language. It supports LP, MILP, NLP, MINLP and
  differential-algebraic models, is solver-agnostic, and is the modeling layer of IDAES and of much of the
  process systems engineering literature. Its cost is that model generation happens in interpreted Python,
  so building very large models can be slower than in a compiled tool, and its error messages can be
  indirect when a model is malformed.

- **JuMP, for large models and for algorithms written around the model.** JuMP is compiled through Julia,
  so the loops that generate a model with millions of terms run at speed comparable to C, and the same
  language is fast enough to write the surrounding algorithm: column generation, Benders decomposition, a
  custom branch-and-price scheme, callbacks inside a solver's search. It has a clean and stable solver
  interface (MathOptInterface) and strong support for nonlinear expressions and automatic differentiation.
  Its cost is the smaller ecosystem, fewer engineers in a typical process company who read Julia, and a
  compilation delay on first use of a session.

- **GAMS, for legacy industrial models and where the licenses already exist.** Planning models in
  refining, petrochemicals and energy have been maintained in GAMS for decades, and those models are
  assets: rewriting a validated model is a project with risk and no immediate payoff. GAMS is stable, its
  syntax has changed little, its listing file is a complete audit trail, and it ships with an unusually
  wide set of commercial solvers behind one interface. Its cost is commercial licensing for both the
  system and the solvers, a language that exists only for this purpose (so data handling, plotting and
  testing must happen outside it or through GDX), and a smaller pool of new graduates who know it. AMPL
  occupies a similar position with a different syntax.

Two remarks that apply to all four. First, none of the four solves anything: HiGHS, CBC, GLPK, Gurobi,
CPLEX, Ipopt and the rest do, and the same solver can usually be reached from all four. Solver choice
affects run time and, for hard MILPs and NLPs, whether an answer is obtained at all; modeling language
choice does not. Second, the expensive part of an industrial optimization project is the model and its
data, not the syntax. That is why porting a model between these tools is routine, and why a specification
written at the level of Section 1 is worth more than any of the four listings.

## Exercises

1. **(Basic) Predict, then solve.** The reactor is offered 10 extra hours per week, raising `b_reactor`
   from 130 to 140. Use the shadow price from Section 3.3 to predict the new optimal margin before
   changing anything. Then re-solve in Pyomo and compare. Next, state which cell you would edit in the
   Section 2 worksheet, which line in the JuMP listing, and which entry in the GAMS `Parameters` block,
   and say which of the four tools you would hand to a plant manager who wants to try ten different values
   himself.

2. **(Intermediate) Add a product.** A fourth product `E4` is proposed with margin `2.8` kUSD/t, resource
   use `(reactor, purification, packaging) = (3, 2, 2)` h/t and the same 25 t/week tankage limit. Write
   down the exact edits required in each of the four tools, counting the number of places each edit has to
   be made. Then implement it in Pyomo and report whether `E4` enters the optimal plan, what happens to
   `z`, and whether the shadow prices change.

3. **(Advanced) Read a model you did not write.** Without looking at Section 1, read only the JuMP listing
   in Section 4 and write out the algebraic model it defines, with a symbol table and units. Compare with
   Section 1 and note anything you could not recover from the code alone. Then add the requirement that
   high-purity electrolyte must be at least 30 percent of total tonnage, `x_E2 >= 0.3 * sum_p x_p`, express
   it in all four tools, and solve it in Pyomo. Report the new plan, the new margin, and which resource
   constraints remain tight.

## Takeaways

- The model is separable from the modeling language. One product mix LP, four tools, one optimum:
  `x = (20, 15, 10)` t/week and `z = 121.5` kUSD/week, with shadow prices `(0.50, 0.40, 0.30)` kUSD/h.
- Six constructs are enough to write any linear or mixed-integer model: index set, parameter, variable
  with bounds, linear constraint over a set, objective, and solve and retrieve. Learning a new modeling
  language means finding those six in its syntax, and nothing more.
- The largest practical difference between the tools is whether the index set is an explicit object.
  Where it is, adding a product is one line of data; in a spreadsheet it is a structural edit.
- The modeling layer never solves anything. All four call the same class of solvers, so the language
  affects convenience, speed of model generation, licensing and auditability, not the optimum.
- Always check the status: `res.solver.termination_condition` in Pyomo, `termination_status` in JuMP,
  `modelstat` in GAMS, the Solver Results dialog in Excel. A reported number without a proven status is
  not a result.
- A uniquely optimal, non-degenerate LP gives a strong cross-tool check: because the optimal basis is
  unique, any correct implementation must return exactly the same vertex, so a disagreement between tools
  is always a transcription error in the model or the data.